[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/13_modules-apis.ipynb)

*Python from Scratch · Notebook 13 of 13*

# Modules, Packages, APIs & JSON

Split code into modules and packages, install libraries in a virtual environment, read and write JSON, and call web APIs the safe way: status codes, pagination, retries, rate limits and caching.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A module is a toolbox in its own drawer, and an API is a shop counter: you ask in a fixed format (JSON), and you get an answer with a status code.

**What interviewers really test here:** __name__ == '__main__', virtual environments, json.loads vs json.load, status codes, retries with backoff, pagination and keeping secrets out of code.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Modules: one file, one job *(Basics)*
- [ ] 2. Packages and the standard library *(Basics)*
- [ ] 3. pip and virtual environments *(Core)*
- [ ] 4. JSON: the language APIs speak *(Core)*
- [ ] 5. Calling a web API *(Core)*
- [ ] 6. Robust API clients: pages, retries, limits, caching *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Module, package, library

- **Module**: one `.py` file. Imported once per program; later imports reuse the cached copy in `sys.modules`.
- **Package**: a folder of modules, usually with `__init__.py`. Dotted names follow folders.
- **Library / distribution**: what you install with pip (one distribution can contain several packages; its pip name can differ from its import name, e.g. `pip install scikit-learn` → `import sklearn`).
- **Standard library**: modules that ship with Python.

### How import finds a module

1. Already in `sys.modules`? Reuse it (that's why module-level code runs only once).
2. Otherwise search `sys.path` in order: the script's folder, `PYTHONPATH`, the standard library, `site-packages`.
3. Run the file top to bottom, and bind the name in the importing module.



`from m import *` imports the names in `m.__all__` if it exists, otherwise every name that doesn't start with `_`. Avoid it outside quick experiments: it hides where names come from.

### HTTP status codes worth knowing

| Code | Meaning | What to do |
| --- | --- | --- |
| 200 / 201 / 204 | OK / created / no content | use the body |
| 301 / 302 / 304 | moved / found elsewhere / not modified | follow, or use your cached copy |
| 400 | bad request | fix your parameters |
| 401 / 403 | not logged in / not allowed | check the token and permissions |
| 404 | not found | don't retry |
| 429 | too many requests | slow down, honour `Retry-After` |
| 500 / 502 / 503 / 504 | server-side failure | retry with backoff |

### Idempotency

A request is **idempotent** if sending it twice has the same effect as once. GET, PUT and DELETE should be; POST usually isn't (two POSTs create two orders). Retrying is only safe for idempotent requests — many payment APIs accept an *idempotency key* so a retried POST isn't applied twice.

---
# Part 2: Lessons, step by step

## Lesson 1: Modules: one file, one job

*Level: Basics*

Once a program grows past a few hundred lines, you split it into **modules**. A module is just a `.py` file, and its name is the file name without `.py`. `import` runs that file once and gives you its names.



```python
# prices.py
TAX = 0.18
def with_tax(amount):
    return round(amount * (1 + TAX), 2)

# main.py
import prices                 # use as prices.with_tax(100)
from prices import with_tax   # use as with_tax(100)
import prices as p            # short alias: p.with_tax(100)
```



Every module has a `__name__`. It is `"__main__"` when you run the file directly, and the module's own name when someone imports it. That's why scripts end with:



```python
if __name__ == "__main__":
    main()        # runs for `python prices.py`, not for `import prices`
```



**Where does Python look?** First the folder of the script you ran, then the standard library, then installed packages (`site-packages`). The list is in `sys.path`. Naming your own file `json.py` or `random.py` hides the real module — a classic beginner bug.

**✏️ Your turn:** you now have what you need for [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 2: Packages and the standard library

*Level: Basics*

A **package** is a folder of modules. Dotted names follow the folders:



```python
app/
    __init__.py        # runs when the package is imported (may be empty)
    api/
        __init__.py
        orders.py      # import app.api.orders
    utils.py           # inside app.api.orders: from ..utils import slugify
```



Relative imports start with dots: one dot is "this package", two dots is "the parent". They only work inside a package, not in a script you run directly.



Python ships with a big **standard library**. Learn where things live before installing anything:



| Need | Module |
| --- | --- |
| JSON, CSV | `json`, `csv` |
| Dates and times | `datetime`, `zoneinfo` |
| Counting, queues, defaults | `collections` (`Counter`, `deque`, `defaultdict`) |
| Files and paths | `pathlib`, `os`, `shutil` |
| URLs and HTTP | `urllib.parse`, `urllib.request` |
| Text patterns | `re` |
| Caching, partials | `functools` |
| Logs instead of prints | `logging` |
| Small databases | `sqlite3` |

**✏️ Your turn:** you now have what you need for [Q20](#q20), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 3: pip and virtual environments

*Level: Core*

Libraries outside the standard library (pandas, requests, scikit-learn) are installed with **pip** from PyPI. If every project installs into one global Python, project A needing pandas 1.5 and project B needing pandas 2.2 will fight. A **virtual environment** gives each project its own private `site-packages`.



```python
python -m venv .venv              # create (once per project)
source .venv/bin/activate         # macOS / Linux   (.venv\Scripts\activate on Windows)
pip install pandas==2.2.3 requests
pip freeze > requirements.txt     # record exact versions
pip install -r requirements.txt   # a teammate recreates the same set
deactivate
```



Rules that save hours:




- Never commit `.venv` to git; commit `requirements.txt` (or `pyproject.toml`).
- Pin versions for anything that runs in production, so a new release can't break you overnight.
- Use `python -m pip` if you are unsure which Python `pip` belongs to.
- Versions compare part by part as numbers: 1.10 is newer than 1.9.

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 4: JSON: the language APIs speak

*Level: Core*

**JSON** is text that looks like Python dicts and lists. Almost every web API sends and receives it.



| JSON | Python |
| --- | --- |
| object `{}` | dict (keys are always strings) |
| array `[]` | list (tuples are written as arrays) |
| string, number | str, int / float |
| `true` / `false` / `null` | `True` / `False` / `None` |



```python
text = json.dumps({"id": 1, "ok": True}, indent=2)   # dump to a string
data = json.loads(text)                               # load from a string
with open("orders.json") as f:
    orders = json.load(f)                             # load / dump (no s) work with files
```



Dates have no JSON type, so APIs send ISO text like `"2026-03-01"`; turn it back with `date.fromisoformat`. Real API data is nested and has gaps, so read deep fields defensively.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
import json


def json_demo():
    order = {"id": 17, "items": ("pen", "book"), "paid": True, "coupon": None}
    text = json.dumps(order)
    back = json.loads(text)
    same = back == order
    items_type = type(back["items"]).__name__


json_demo()

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q12](#q12), [Q3](#q3), [Q1](#q1), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 5: Calling a web API

*Level: Core*

An **API** is a URL that returns data instead of a web page. You send an HTTP **request** and get a **response** with a status code, headers and a body (usually JSON).



```python
import os, requests

r = requests.get(
    "https://api.example.com/v1/orders",
    params={"status": "paid", "limit": 50},                     # becomes ?status=paid&limit=50
    headers={"Authorization": f"Bearer {os.environ['API_TOKEN']}"},
    timeout=10,                                                 # never wait forever
)
r.raise_for_status()      # turns 4xx / 5xx into an exception
orders = r.json()
```



| Method | Meaning |
| --- | --- |
| GET | read |
| POST | create (send JSON with `json=...`) |
| PUT / PATCH | replace / update part |
| DELETE | remove |



Keep secrets such as tokens in environment variables, never in code or notebooks you share. The browser IDE can't make real network calls, so the practice questions fake the server with plain data — the logic is identical.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
from urllib.parse import urlencode, quote


def query_demo():
    params = {"q": "data science", "page": 2}
    query = urlencode(params, quote_via=quote)
    url = "https://api.example.com/search?" + query


query_demo()

**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q14](#q14), [Q5](#q5), [Q6](#q6), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 6: Robust API clients: pages, retries, limits, caching

*Level: Advanced*

Real APIs fail and limit you. A production client handles four things:




1. **Pagination.** Big results come in pages. Follow the `next` cursor (or page number) until there isn't one.
2. **Retries with backoff.** 429 (too many requests) and 5xx (server trouble) are often temporary: wait 1 s, 2 s, 4 s… with a cap, plus a little random jitter so many clients don't retry together. 4xx errors like 404 won't fix themselves, so don't retry them.
3. **Rate limits.** Respect "N requests per window" yourself instead of waiting for 429s. A sliding window keeps the timestamps of recent calls.
4. **Caching.** Don't fetch the same thing twice: `functools.lru_cache` for one run, a file or database cache across runs.



```python
for attempt in range(5):
    r = requests.get(url, timeout=10)
    if r.status_code != 429 and r.status_code < 500:
        break
    time.sleep(min(30, 2 ** attempt) + random.random())
```

**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q7](#q7), [Q15](#q15), [Q18](#q18), [Q19](#q19). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [4]:
import json

from collections import Counter, deque

from urllib.parse import urlencode, quote, urlparse, parse_qs


<a id="q1"></a>
### Q1. Read a Nested JSON Field

🟢 Warm-up · *Nested dicts and lists*

Follow `path` (a list of keys and list indexes) into parsed JSON. Return `default` if any step is missing.

**Examples**

- `data = {'user': {'tags': ['a', 'b']}}, path = ['user', 'tags', 1], default = None` → `'b'`
- `data = {'user': {'tags': []}}, path = ['user', 'tags', 0], default = 'none'` → `'none'`
- `data = {'user': None}, path = ['user', 'name'], default = '?'` → `'?'`

In [ ]:
def dig(data, path, default):
    """
    Follow path (a list of keys and list indexes) into parsed JSON. Return
    default if any step is missing.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dig, [
    (({'user': {'tags': ['a', 'b']}}, ['user', 'tags', 1], None), 'b'),
    (({'user': {'tags': []}}, ['user', 'tags', 0], 'none'), 'none'),
    (({'user': None}, ['user', 'name'], '?'), '?'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check the type and the key or index at every step before stepping in.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Just try every step and catch the three errors a missing step can raise.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check the type and the key or index at every step before stepping in.

```python
def dig(data, path, default):
    cur = data
    for step in path:
        if isinstance(cur, dict) and step in cur:
            cur = cur[step]
        elif isinstance(cur, list) and isinstance(step, int) and -len(cur) <= step < len(cur):
            cur = cur[step]
        else:
            return default
    return cur
```

**Cost:** O(depth) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Just try every step and catch the three errors a missing step can raise.

```python
def dig(data, path, default):
    cur = data
    try:
        for step in path:
            cur = cur[step]
    except (KeyError, IndexError, TypeError):
        return default
    return cur
```

**Cost:** O(depth) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check the type and the key or index at every step before stepping in. That costs O(depth) time and O(1) space."

"I can do better: Just try every step and catch the three errors a missing step can raise. That's O(depth) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/read-a-nested-json-field)

<a id="q2"></a>
### Q2. JSON Text for a Flat Record

🟢 Warm-up · *json.dumps*

Turn a flat dict of strings, ints and booleans into compact JSON text with keys sorted, like `{"a": 1, "b": true}`.

**Examples**

- `record = {'name': 'Asha', 'age': 31, 'admin': False}` → `'{"admin": false, "age": 31, "name": "Asha"}'`
- `record = {}` → `'{}'`

In [ ]:
def to_json(record):
    """
    Turn a flat dict of strings, ints and booleans into compact JSON text with
    keys sorted, like {"a": 1, "b": true}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_json, [
    (({'name': 'Asha', 'age': 31, 'admin': False},), '{"admin": false, "age": 31, "name": "Asha"}'),
    (({},), '{}'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build the text yourself: quote strings, lower-case booleans, join with commas.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`json.dumps(record, sort_keys=True)` already knows every rule, including escaping.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Build the text yourself: quote strings, lower-case booleans, join with commas.

```python
def to_json(record):
    parts = []
    for key in sorted(record):
        value = record[key]
        if isinstance(value, bool):
            text = "true" if value else "false"
        elif isinstance(value, str):
            text = '"' + value + '"'
        else:
            text = str(value)
        parts.append('"' + key + '": ' + text)
    return "{" + ", ".join(parts) + "}"
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `json.dumps(record, sort_keys=True)` already knows every rule, including escaping.

```python
def to_json(record):
    return json.dumps(record, sort_keys=True)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build the text yourself: quote strings, lower-case booleans, join with commas. That costs O(n log n) time and O(n) space."

"I can do better: `json.dumps(record, sort_keys=True)` already knows every rule, including escaping. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/json-text-for-a-flat-record)

<a id="q3"></a>
### Q3. Count Records by Status

🟢 Warm-up · *json.loads + counting*

`text` is a JSON array of objects that each have a `status`. Return a dict of status → count, sorted by status.

**Examples**

- `text = '[{"id":1,"status":"paid"},{"id":2,"status":"failed"},{"id":3,"status":"paid"}]'` → `{'failed': 1, 'paid': 2}`
- `text = '[]'` → `{}`

In [ ]:
def status_counts(text):
    """
    text is a JSON array of objects that each have a status. Return a dict of
    status → count, sorted by status.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(status_counts, [
    (('[{"id":1,"status":"paid"},{"id":2,"status":"failed"},{"id":3,"status":"paid"}]',), {'failed': 1, 'paid': 2}),
    (('[]',), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Parse, then count in a dictionary with `get`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Parse, then `Counter`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Parse, then count in a dictionary with `get`.

```python
def status_counts(text):
    records = json.loads(text)
    counts = {}
    for r in records:
        counts[r["status"]] = counts.get(r["status"], 0) + 1
    return {k: counts[k] for k in sorted(counts)}
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Parse, then `Counter`.

```python
def status_counts(text):
    counts = Counter(r["status"] for r in json.loads(text))
    return dict(sorted(counts.items()))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Parse, then count in a dictionary with `get`. That costs O(n log n) time and O(n) space."

"I can do better: Parse, then `Counter`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/count-records-by-status)

<a id="q4"></a>
### Q4. Build a Query String

🟢 Warm-up · *URL encoding*

Turn a dict of parameters into a query string. Spaces become `%20` and characters outside letters, digits and `-_.~` are percent-encoded.

**Examples**

- `params = {'q': 'data science', 'page': 2}` → `'q=data%20science&page=2'`
- `params = {'city': 'São Paulo', 'sort': '-date'}` → `'city=S%C3%A3o%20Paulo&sort=-date'`

In [ ]:
def query_string(params):
    """
    Turn a dict of parameters into a query string. Spaces become %20 and
    characters outside letters, digits and -_.~ are percent-encoded.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(query_string, [
    (({'q': 'data science', 'page': 2},), 'q=data%20science&page=2'),
    (({'city': 'São Paulo', 'sort': '-date'},), 'city=S%C3%A3o%20Paulo&sort=-date'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Encode each character yourself: UTF-8 bytes, then `%XX` for anything unsafe.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`urlencode(params, quote_via=quote)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Encode each character yourself: UTF-8 bytes, then `%XX` for anything unsafe.

```python
def query_string(params):
    safe = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_.~"

    def enc(text):
        out = ""
        for byte in str(text).encode("utf-8"):
            ch = chr(byte)
            out += ch if ch in safe else "%" + format(byte, "02X")
        return out

    return "&".join(enc(k) + "=" + enc(v) for k, v in params.items())
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `urlencode(params, quote_via=quote)`.

```python
def query_string(params):
    return urlencode(params, quote_via=quote)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Encode each character yourself: UTF-8 bytes, then `%XX` for anything unsafe. That costs O(n) time and O(n) space."

"I can do better: `urlencode(params, quote_via=quote)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/build-a-query-string)

<a id="q5"></a>
### Q5. Join API Paths Without Double Slashes

🟢 Warm-up · *Building endpoints*

Join a base URL and path parts with exactly one `/` between them.

**Examples**

- `base = 'https://api.x.com/v1/', parts = ['/users/', '42']` → `'https://api.x.com/v1/users/42'`
- `base = 'https://api.x.com', parts = ['orders']` → `'https://api.x.com/orders'`

In [ ]:
def join_url(base, parts):
    """
    Join a base URL and path parts with exactly one / between them.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(join_url, [
    (('https://api.x.com/v1/', ['/users/', '42']), 'https://api.x.com/v1/users/42'),
    (('https://api.x.com', ['orders']), 'https://api.x.com/orders'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Trim slashes at the joins one part at a time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Strip every part and `'/'.join` once.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Trim slashes at the joins one part at a time.

```python
def join_url(base, parts):
    url = base
    for part in parts:
        while url.endswith("/"):
            url = url[:-1]
        while part.startswith("/"):
            part = part[1:]
        while part.endswith("/"):
            part = part[:-1]
        url = url + "/" + part
    return url
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Strip every part and `'/'.join` once.

```python
def join_url(base, parts):
    return "/".join([base.rstrip("/")] + [p.strip("/") for p in parts])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Trim slashes at the joins one part at a time. That costs O(n) time and O(n) space."

"I can do better: Strip every part and `'/'.join` once. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/join-api-paths-without-double-slashes)

<a id="q6"></a>
### Q6. Classify an HTTP Status Code

🟢 Warm-up · *Status families*

Return 'ok' for 2xx, 'redirect' for 3xx, 'retry' for 429 or any 5xx, 'client error' for other 4xx, else 'unknown'.

**Examples**

- `code = 200` → `'ok'`
- `code = 201` → `'ok'`
- `code = 304` → `'redirect'`

In [ ]:
def status_kind(code):
    """
    Return 'ok' for 2xx, 'redirect' for 3xx, 'retry' for 429 or any 5xx, 'client
    error' for other 4xx, else 'unknown'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(status_kind, [
    ((200,), 'ok'),
    ((201,), 'ok'),
    ((304,), 'redirect'),
    ((404,), 'client error'),
    ((429,), 'retry'),
    ((503,), 'retry'),
    ((99,), 'unknown'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if-chain over ranges.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Special-case 429, then look up the family `code // 100` in a dict.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** An if-chain over ranges.

```python
def status_kind(code):
    if 200 <= code <= 299:
        return "ok"
    if 300 <= code <= 399:
        return "redirect"
    if code == 429:
        return "retry"
    if 400 <= code <= 499:
        return "client error"
    if 500 <= code <= 599:
        return "retry"
    return "unknown"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Special-case 429, then look up the family `code // 100` in a dict.

```python
def status_kind(code):
    if code == 429:
        return "retry"
    families = {2: "ok", 3: "redirect", 4: "client error", 5: "retry"}
    return families.get(code // 100, "unknown")
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if-chain over ranges. That costs O(1) time and O(1) space."

"I can do better: Special-case 429, then look up the family `code // 100` in a dict. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/classify-an-http-status-code)

<a id="q7"></a>
### Q7. Exponential Backoff Delays

🟢 Warm-up · *Retry timing*

Return the wait (seconds) before each of `tries` retries: `base × 2^i`, never more than `cap`.

**Examples**

- `base = 1, tries = 5, cap = 10` → `[1, 2, 4, 8, 10]`
- `base = 0.5, tries = 4, cap = 100` → `[0.5, 1.0, 2.0, 4.0]`
- `base = 2, tries = 0, cap = 10` → `[]`

In [ ]:
def backoff(base, tries, cap):
    """
    Return the wait (seconds) before each of tries retries: base × 2^i, never
    more than cap.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(backoff, [
    ((1, 5, 10), [1, 2, 4, 8, 10]),
    ((0.5, 4, 100), [0.5, 1.0, 2.0, 4.0]),
    ((2, 0, 10), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop that doubles a running delay and clips it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One comprehension with `min`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A loop that doubles a running delay and clips it.

```python
def backoff(base, tries, cap):
    delays = []
    d = base
    for _ in range(tries):
        delays.append(d if d < cap else cap)
        d = d * 2
    return delays
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** One comprehension with `min`.

```python
def backoff(base, tries, cap):
    return [min(cap, base * 2 ** i) for i in range(tries)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop that doubles a running delay and clips it. That costs O(n) time and O(n) space."

"I can do better: One comprehension with `min`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/exponential-backoff-delays)

<a id="q8"></a>
### Q8. Days Between ISO Dates

🟢 Warm-up · *datetime from JSON*

APIs send dates as ISO text like `"2026-03-01"`. Return the number of days from `start` to `end`.

**Examples**

- `start = '2026-02-27', end = '2026-03-01'` → `2`
- `start = '2024-02-28', end = '2024-03-01'` → `2`
- `start = '2026-01-10', end = '2026-01-10'` → `0`

In [ ]:
from datetime import date

def days_between(start, end):
    """
    APIs send dates as ISO text like "2026-03-01". Return the number of days
    from start to end.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(days_between, [
    (('2026-02-27', '2026-03-01'), 2),
    (('2024-02-28', '2024-03-01'), 2),
    (('2026-01-10', '2026-01-10'), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count days by hand: days in each month, with the leap-year rule.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`date.fromisoformat`, then subtract: a `timedelta` knows its `.days`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Count days by hand: days in each month, with the leap-year rule.

```python
def days_between(start, end):
    def day_number(text):
        y, m, d = (int(p) for p in text.split("-"))
        months = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
        total = 0
        for year in range(1, y):
            leap = year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)
            total += 366 if leap else 365
        leap = y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)
        for month in range(1, m):
            total += 29 if month == 2 and leap else months[month - 1]
        return total + d
    return day_number(end) - day_number(start)
```

**Cost:** O(years) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `date.fromisoformat`, then subtract: a `timedelta` knows its `.days`.

```python
def days_between(start, end):
    return (date.fromisoformat(end) - date.fromisoformat(start)).days
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count days by hand: days in each month, with the leap-year rule. That costs O(years) time and O(1) space."

"I can do better: `date.fromisoformat`, then subtract: a `timedelta` knows its `.days`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/days-between-iso-dates)

<a id="q9"></a>
### Q9. What Does import \* Bring In?

🟢 Warm-up · *\_\_all\_\_ and private names*

Given a module's top-level names and its `__all__` (or None), return the names `from module import *` would import, sorted. Without `__all__`, names starting with `_` are skipped.

**Examples**

- `names = ['load', 'save', '_cache', 'VERSION'], all_list = None` → `['VERSION', 'load', 'save']`
- `names = ['load', 'save', '_cache'], all_list = ['load']` → `['load']`

In [ ]:
def star_names(names, all_list):
    """
    Given a module's top-level names and its __all__ (or None), return the names
    from module import * would import, sorted. Without __all__, names starting
    with _ are skipped.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(star_names, [
    ((['load', 'save', '_cache', 'VERSION'], None), ['VERSION', 'load', 'save']),
    ((['load', 'save', '_cache'], ['load']), ['load']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two branches with a loop each.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One expression: `__all__` if given, else a filtered comprehension.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Two branches with a loop each.

```python
def star_names(names, all_list):
    out = []
    if all_list is not None:
        for n in all_list:
            out.append(n)
    else:
        for n in names:
            if not n.startswith("_"):
                out.append(n)
    return sorted(out)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** One expression: `__all__` if given, else a filtered comprehension.

```python
def star_names(names, all_list):
    return sorted(all_list if all_list is not None else [n for n in names if not n.startswith("_")])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two branches with a loop each. That costs O(n log n) time and O(n) space."

"I can do better: One expression: `__all__` if given, else a filtered comprehension. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/what-does-import-bring-in)

<a id="q10"></a>
### Q10. Read a requirements.txt

🟢 Warm-up · *Pinned dependencies*

Return a dict of package → version spec (like `'==2.2.0'`, or `''` if unpinned). Skip blank lines and comments; package names are lower-cased.

**Examples**

- `text = '# web\nrequests==2.32.3\n\nPandas>=2.2\nnumpy  # latest\n'` → `{'requests': '==2.32.3', 'pandas': '>=2.2', 'numpy': ''}`

In [ ]:
import re

def requirements(text):
    """
    Return a dict of package → version spec (like '==2.2.0', or '' if unpinned).
    Skip blank lines and comments; package names are lower-cased.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(requirements, [
    (('# web\nrequests==2.32.3\n\nPandas>=2.2\nnumpy  # latest\n',), {'requests': '==2.32.3', 'pandas': '>=2.2', 'numpy': ''}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Strip comments, then scan each line for the first of `= < > ! ~`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One regular expression: a name, then an optional operator and version.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Strip comments, then scan each line for the first of `= < > ! ~`.

```python
def requirements(text):
    out = {}
    for line in text.split("\n"):
        line = line.split("#")[0].strip()
        if not line:
            continue
        cut = len(line)
        for i, ch in enumerate(line):
            if ch in "=<>!~":
                cut = i
                break
        out[line[:cut].strip().lower()] = line[cut:].replace(" ", "")
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** One regular expression: a name, then an optional operator and version.

```python
def requirements(text):
    out = {}
    for line in text.splitlines():
        m = re.match(r"\s*([A-Za-z0-9_.\-]+)\s*([=<>!~].*)?$", line.split("#")[0])
        if m:
            out[m.group(1).lower()] = (m.group(2) or "").replace(" ", "")
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Strip comments, then scan each line for the first of `= < > ! ~`. That costs O(n) time and O(n) space."

"I can do better: One regular expression: a name, then an optional operator and version. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/read-a-requirements-txt)

<a id="q11"></a>
### Q11. Compare Version Numbers

🟢 Warm-up · *Versions are not strings*

Return −1, 0 or 1 comparing versions like `'1.10.0'` and `'1.9'` part by part (missing parts count as 0).

**Examples**

- `a = '1.10.0', b = '1.9'` → `1`
- `a = '2.0', b = '2.0.0'` → `0`
- `a = '0.9.9', b = '1.0'` → `-1`

In [ ]:
def compare_versions(a, b):
    """
    Return −1, 0 or 1 comparing versions like '1.10.0' and '1.9' part by part
    (missing parts count as 0).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compare_versions, [
    (('1.10.0', '1.9'), 1),
    (('2.0', '2.0.0'), 0),
    (('0.9.9', '1.0'), -1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk both lists of parts, padding the shorter with zeros.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Pad to equal length and compare tuples of ints; `(a > b) - (a < b)` gives the sign.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Walk both lists of parts, padding the shorter with zeros.

```python
def compare_versions(a, b):
    pa = a.split(".")
    pb = b.split(".")
    for i in range(max(len(pa), len(pb))):
        x = int(pa[i]) if i < len(pa) else 0
        y = int(pb[i]) if i < len(pb) else 0
        if x > y:
            return 1
        if x < y:
            return -1
    return 0
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Pad to equal length and compare tuples of ints; `(a > b) - (a < b)` gives the sign.

```python
def compare_versions(a, b):
    pa, pb = [list(map(int, v.split("."))) for v in (a, b)]
    n = max(len(pa), len(pb))
    ta, tb = tuple(pa + [0] * (n - len(pa))), tuple(pb + [0] * (n - len(pb)))
    return (ta > tb) - (ta < tb)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk both lists of parts, padding the shorter with zeros. That costs O(n) time and O(n) space."

"I can do better: Pad to equal length and compare tuples of ints; `(a > b) - (a < b)` gives the sign. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/compare-version-numbers)

<a id="q12"></a>
### Q12. What Survives a JSON Round Trip?

🟡 Interview standard · *Type mapping*

Return what you get back after `json.loads(json.dumps(x))`: tuples become lists and dict keys become strings.

**Examples**

- `x = {1: (2, 3), 'x': [True, None]}` → `{'1': [2, 3], 'x': [True, None]}`
- `x = (1, ('a', 2.5))` → `[1, ['a', 2.5]]`

In [ ]:
def round_trip(x):
    """
    Return what you get back after json.loads(json.dumps(x)): tuples become
    lists and dict keys become strings.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(round_trip, [
    (({1: (2, 3), 'x': [True, None]},), {'1': [2, 3], 'x': [True, None]}),
    (((1, ('a', 2.5)),), [1, ['a', 2.5]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the value recursively: tuples to lists, every key to `str` (with True → 'true').

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Do the round trip itself.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Walk the value recursively: tuples to lists, every key to `str` (with True → 'true').

```python
def round_trip(x):
    if isinstance(x, dict):
        out = {}
        for k, v in x.items():
            if isinstance(k, bool):
                key = "true" if k else "false"
            elif k is None:
                key = "null"
            else:
                key = str(k)
            out[key] = round_trip(v)
        return out
    if isinstance(x, (list, tuple)):
        return [round_trip(v) for v in x]
    return x
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Do the round trip itself.

```python
def round_trip(x):
    return json.loads(json.dumps(x))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the value recursively: tuples to lists, every key to `str` (with True → 'true'). That costs O(n) time and O(n) space."

"I can do better: Do the round trip itself. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/what-survives-a-json-round-trip)

<a id="q13"></a>
### Q13. Flatten Nested JSON

🟡 Interview standard · *Recursion · dotted keys*

Flatten nested dicts into one level with dotted keys: `{"a": {"b": 1}}` → `{"a.b": 1}`. Lists stay as values.

**Examples**

- `obj = {'id': 7, 'user': {'name': 'Ravi', 'address': {'city': 'Pune'}}, 'tags': ['x']}` → `{'id': 7, 'tags': ['x'], 'user.address.city': 'Pune', 'user.name': 'Ravi'}`
- `obj = {'a': {}}` → `{}`

In [ ]:
def flatten(obj, prefix=''):
    """
    Flatten nested dicts into one level with dotted keys: {"a": {"b": 1}} →
    {"a.b": 1}. Lists stay as values.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(flatten, [
    (({'id': 7, 'user': {'name': 'Ravi', 'address': {'city': 'Pune'}}, 'tags': ['x']},), {'id': 7, 'tags': ['x'], 'user.address.city': 'Pune', 'user.name': 'Ravi'}),
    (({'a': {}},), {}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An explicit stack of (prefix, dict) pairs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A short recursive function that passes the prefix down.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** An explicit stack of (prefix, dict) pairs.

```python
def flatten(obj):
    out = {}
    stack = [("", obj)]
    while stack:
        prefix, d = stack.pop()
        for k, v in d.items():
            key = prefix + "." + k if prefix else k
            if isinstance(v, dict):
                stack.append((key, v))
            else:
                out[key] = v
    return dict(sorted(out.items()))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A short recursive function that passes the prefix down.

```python
def flatten(obj, prefix=""):
    out = {}
    for k, v in obj.items():
        key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            out.update(flatten(v, key))
        else:
            out[key] = v
    return dict(sorted(out.items()))
```

**Cost:** O(n) time, O(depth) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An explicit stack of (prefix, dict) pairs. That costs O(n) time and O(n) space."

"I can do better: A short recursive function that passes the prefix down. That's O(n) time and O(depth) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `obj = {'id': 7, 'user': {'name': 'Ravi', 'address': {'city': 'Pune'}}}`

1. flatten() returns {'user.address.city': 'Pune'}
2. flatten() returns {'user.address.city': 'Pune', 'user.n...
3. flatten() returns {'id': 7, 'user.address.city': 'Pune'...

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/flatten-nested-json)

<a id="q14"></a>
### Q14. Pull Apart a URL

🟡 Interview standard · *urllib.parse*

Return `(scheme, host, path, params)` where params maps each query key to its first value.

**Examples**

- `url = 'https://api.shop.in/v2/orders?status=paid&limit=50'` → `('https', 'api.shop.in', '/v2/orders', {'status': 'paid', 'limit': '50'})`
- `url = 'http://localhost:8000/health'` → `('http', 'localhost:8000', '/health', {})`

In [ ]:
def parse_url(url):
    """
    Return (scheme, host, path, params) where params maps each query key to its
    first value.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(parse_url, [
    (('https://api.shop.in/v2/orders?status=paid&limit=50',), ('https', 'api.shop.in', '/v2/orders', {'status': 'paid', 'limit': '50'})),
    (('http://localhost:8000/health',), ('http', 'localhost:8000', '/health', {})),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split on `://`, then the first `/`, then `?`, `&` and `=`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`urlparse` and `parse_qs`, which also handle ports, fragments and encoding.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Split on `://`, then the first `/`, then `?`, `&` and `=`.

```python
def parse_url(url):
    scheme, rest = url.split("://", 1)
    rest, _, query = rest.partition("?")
    host, slash, path = rest.partition("/")
    params = {}
    if query:
        for pair in query.split("&"):
            k, _, v = pair.partition("=")
            params.setdefault(k, v)
    return (scheme, host, slash + path, params)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `urlparse` and `parse_qs`, which also handle ports, fragments and encoding.

```python
def parse_url(url):
    p = urlparse(url)
    params = {k: v[0] for k, v in parse_qs(p.query).items()}
    return (p.scheme, p.netloc, p.path, params)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split on `://`, then the first `/`, then `?`, `&` and `=`. That costs O(n) time and O(n) space."

"I can do better: `urlparse` and `parse_qs`, which also handle ports, fragments and encoding. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/pull-apart-a-url)

<a id="q15"></a>
### Q15. Call With Retries

🟡 Interview standard · *Retry loop*

`responses` is the list of status codes the server will send, in order. Keep calling while the answer is retryable (429 or 5xx) and you have attempts left. Return `(final status, attempts used)`.

**Examples**

- `responses = [503, 503, 200], max_attempts = 5` → `(200, 3)`
- `responses = [500, 500, 500, 500], max_attempts = 3` → `(500, 3)`
- `responses = [404, 200], max_attempts = 3` → `(404, 1)`

In [ ]:
def call_with_retries(responses, max_attempts):
    """
    responses is the list of status codes the server will send, in order. Keep
    calling while the answer is retryable (429 or 5xx) and you have attempts
    left. Return (final status, attempts used).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(call_with_retries, [
    (([503, 503, 200], 5), (200, 3)),
    (([500, 500, 500, 500], 3), (500, 3)),
    (([404, 200], 3), (404, 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A while loop with a counter and a flag.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A for loop over attempts that returns as soon as the answer is final.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A while loop with a counter and a flag.

```python
def call_with_retries(responses, max_attempts):
    attempts = 0
    status = None
    done = False
    while not done:
        status = responses[attempts]
        attempts += 1
        retryable = status == 429 or 500 <= status <= 599
        if not retryable or attempts == max_attempts:
            done = True
    return (status, attempts)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A for loop over attempts that returns as soon as the answer is final.

```python
def call_with_retries(responses, max_attempts):
    for attempt in range(1, max_attempts + 1):
        status = responses[attempt - 1]
        if not (status == 429 or status >= 500):
            break
    return (status, attempt)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A while loop with a counter and a flag. That costs O(n) time and O(1) space."

"I can do better: A for loop over attempts that returns as soon as the answer is final. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/call-with-retries)

<a id="q16"></a>
### Q16. Collect Every Page

🟡 Interview standard · *Cursor pagination*

`pages` fakes an API: `pages[cursor]` returns `{"items": [...], "next": cursor or None}`. Start from cursor None and return all items in order.

**Examples**

- `pages = {None: {'items': [1, 2, 3], 'next': 'c2'}, 'c2': {'items': [4, 5], 'next': 'c3'}, 'c3': {'items': [6], 'next': None}}` → `[1, 2, 3, 4, 5, 6]`
- `pages = {None: {'items': [], 'next': None}}` → `[]`

In [ ]:
def all_items(pages):
    """
    pages fakes an API: pages[cursor] returns {"items": [...], "next": cursor or
    None}. Start from cursor None and return all items in order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(all_items, [
    (({None: {'items': [1, 2, 3], 'next': 'c2'}, 'c2': {'items': [4, 5], 'next': 'c3'}, 'c3': {'items': [6], 'next': None}},), [1, 2, 3, 4, 5, 6]),
    (({None: {'items': [], 'next': None}},), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A while-True loop that breaks when there is no next cursor.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A generator that yields items page by page, so a caller could stop early.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A while-True loop that breaks when there is no next cursor.

```python
def all_items(pages):
    items = []
    cursor = None
    while True:
        page = pages[cursor]
        for item in page["items"]:
            items.append(item)
        if page["next"] is None:
            break
        cursor = page["next"]
    return items
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A generator that yields items page by page, so a caller could stop early.

```python
def all_items(pages):
    def walk():
        cursor = None
        while True:
            page = pages[cursor]
            yield from page["items"]
            cursor = page["next"]
            if cursor is None:
                return
    return list(walk())
```

**Cost:** O(n) time, O(1) per page space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A while-True loop that breaks when there is no next cursor. That costs O(n) time and O(n) space."

"I can do better: A generator that yields items page by page, so a caller could stop early. That's O(n) time and O(1) per page space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `pages = {None: {'items': [1, 2, 3], 'next': 'c2'}, 'c2': {'items': [4, 5], 'next': 'c3'}, 'c3': {'items': [6], 'next': None}}`

1. walk() returns 1
2. walk() returns 2
3. walk() returns 3
4. walk() returns 4
5. walk() returns 5
6. walk() returns 6
7. walk() returns None
8. all\_items() returns [1, 2, 3, 4, 5, 6]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/collect-every-page)

<a id="q17"></a>
### Q17. Validate an API Response

🟡 Interview standard · *Schema checks*

`schema` maps each required field to a type name ('str', 'int', 'list'). Return a sorted list of problems: `'missing: f'` or `'wrong type: f'`.

**Examples**

- `payload = {'id': 5, 'name': 'Mina', 'tags': 'x'}, schema = {'id': 'int', 'name': 'str', 'tags': 'list', 'email': 'str'}` → `['missing: email', 'wrong type: tags']`
- `payload = {'id': 1}, schema = {'id': 'int'}` → `[]`

In [ ]:
def validate(payload, schema):
    """
    schema maps each required field to a type name ('str', 'int', 'list').
    Return a sorted list of problems: 'missing: f' or 'wrong type: f'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(validate, [
    (({'id': 5, 'name': 'Mina', 'tags': 'x'}, {'id': 'int', 'name': 'str', 'tags': 'list', 'email': 'str'}), ['missing: email', 'wrong type: tags']),
    (({'id': 1}, {'id': 'int'}), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over the schema and append messages.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Map type names to classes and build both lists with comprehensions.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over the schema and append messages.

```python
def validate(payload, schema):
    problems = []
    for field in schema:
        if field not in payload:
            problems.append("missing: " + field)
        else:
            value = payload[field]
            kind = schema[field]
            ok = (kind == "str" and isinstance(value, str)) or (kind == "int" and isinstance(value, int)) \
                or (kind == "list" and isinstance(value, list))
            if not ok:
                problems.append("wrong type: " + field)
    return sorted(problems)
```

**Cost:** O(k log k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Map type names to classes and build both lists with comprehensions.

```python
def validate(payload, schema):
    types = {"str": str, "int": int, "list": list}
    missing = [f"missing: {f}" for f in schema if f not in payload]
    wrong = [f"wrong type: {f}" for f, t in schema.items() if f in payload and not isinstance(payload[f], types[t])]
    return sorted(missing + wrong)
```

**Cost:** O(k log k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over the schema and append messages. That costs O(k log k) time and O(k) space."

"I can do better: Map type names to classes and build both lists with comprehensions. That's O(k log k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/validate-an-api-response)

<a id="q18"></a>
### Q18. Sliding-Window Rate Limiter

🟡 Interview standard · *Rate limits · deque*

Allow at most `limit` requests in any `window` seconds. Given request times (sorted), return True/False for each: was it allowed? Rejected requests don't count.

**Examples**

- `times = [0, 1, 2, 3, 10, 11], limit = 3, window = 10` → `[True, True, True, False, True, True]`
- `times = [0, 0, 0], limit = 2, window = 1` → `[True, True, False]`

In [ ]:
def allowed(times, limit, window):
    """
    Allow at most limit requests in any window seconds. Given request times
    (sorted), return True/False for each: was it allowed? Rejected requests
    don't count.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(allowed, [
    (([0, 1, 2, 3, 10, 11], 3, 10), [True, True, True, False, True, True]),
    (([0, 0, 0], 2, 1), [True, True, False]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each request, count earlier allowed requests still inside the window.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep allowed times in a deque and drop old ones from the left.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each request, count earlier allowed requests still inside the window.

```python
def allowed(times, limit, window):
    accepted = []
    result = []
    for t in times:
        recent = 0
        for a in accepted:
            if t - a < window:
                recent += 1
        if recent < limit:
            accepted.append(t)
            result.append(True)
        else:
            result.append(False)
    return result
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Keep allowed times in a deque and drop old ones from the left.

```python
def allowed(times, limit, window):
    recent = deque()
    result = []
    for t in times:
        while recent and t - recent[0] >= window:
            recent.popleft()
        ok = len(recent) < limit
        if ok:
            recent.append(t)
        result.append(ok)
    return result
```

**Cost:** O(n) time, O(limit) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each request, count earlier allowed requests still inside the window. That costs O(n²) time and O(n) space."

"I can do better: Keep allowed times in a deque and drop old ones from the left. That's O(n) time and O(limit) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/sliding-window-rate-limiter)

<a id="q19"></a>
### Q19. Cache Repeated API Calls

🟡 Interview standard · *Memoisation · lru\_cache*

A slow `fetch(user_id)` is called for each id in `ids`. With a cache, how many real calls happen? Return `(real calls, results)` where each result is `'user-<id>'`.

**Examples**

- `ids = [3, 1, 3, 3, 2, 1]` → `(3, ['user-3', 'user-1', 'user-3', 'user-3', 'user-2', 'user-1'])`
- `ids = []` → `(0, [])`

In [ ]:
from functools import lru_cache

def cached_calls(ids):
    """
    A slow fetch(user_id) is called for each id in ids. With a cache, how many
    real calls happen? Return (real calls, results) where each result is
    'user-<id>'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cached_calls, [
    (([3, 1, 3, 3, 2, 1],), (3, ['user-3', 'user-1', 'user-3', 'user-3', 'user-2', 'user-1'])),
    (([],), (0, [])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dict cache checked before each call.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Wrap the function with `functools.lru_cache` and read its hit/miss info.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A dict cache checked before each call.

```python
def cached_calls(ids):
    real = 0
    cache = {}
    results = []
    for i in ids:
        if i not in cache:
            real += 1
            cache[i] = "user-" + str(i)
        results.append(cache[i])
    return (real, results)
```

**Cost:** O(n) time, O(unique) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Wrap the function with `functools.lru_cache` and read its hit/miss info.

```python
def cached_calls(ids):
    @lru_cache(maxsize=None)
    def fetch(user_id):
        return f"user-{user_id}"
    results = [fetch(i) for i in ids]
    return (fetch.cache_info().misses, results)
```

**Cost:** O(n) time, O(unique) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dict cache checked before each call. That costs O(n) time and O(unique) space."

"I can do better: Wrap the function with `functools.lru_cache` and read its hit/miss info. That's O(n) time and O(unique) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `ids = [3, 1, 3, 3, 2, 1]`

1. fetch() returns 'user-3'
2. fetch() returns 'user-1'
3. fetch() returns 'user-2'
4. cached\_calls() returns (3, ['user-3', 'user-1', 'user-3', 'u...

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/cache-repeated-api-calls)

<a id="q20"></a>
### Q20. Resolve a Relative Import

🟡 Interview standard · *Packages*

Inside package `package`, resolve a relative import like `'..utils'` to its absolute name. One dot is the current package; each extra dot goes up a level.

**Examples**

- `name = '..utils', package = 'app.api.v1'` → `'app.api.utils'`
- `name = '.models', package = 'app.api'` → `'app.api.models'`
- `name = '...core.db', package = 'app.api.v1'` → `'app.core.db'`

In [ ]:
import importlib.util

def resolve(name, package):
    """
    Inside package package, resolve a relative import like '..utils' to its
    absolute name. One dot is the current package; each extra dot goes up a
    level.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(resolve, [
    (('..utils', 'app.api.v1'), 'app.api.utils'),
    (('.models', 'app.api'), 'app.api.models'),
    (('...core.db', 'app.api.v1'), 'app.core.db'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count leading dots, drop that many − 1 parts from the package, then add the rest.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`importlib.util.resolve_name` does exactly what `import` does.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Count leading dots, drop that many − 1 parts from the package, then add the rest.

```python
def resolve(name, package):
    dots = len(name) - len(name.lstrip("."))
    rest = name[dots:]
    parts = package.split(".")
    base = parts[:len(parts) - (dots - 1)]
    return ".".join(base + ([rest] if rest else []))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `importlib.util.resolve_name` does exactly what `import` does.

```python
def resolve(name, package):
    return importlib.util.resolve_name(name, package)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count leading dots, drop that many − 1 parts from the package, then add the rest. That costs O(n) time and O(n) space."

"I can do better: `importlib.util.resolve_name` does exactly what `import` does. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/modules-apis/resolve-a-relative-import)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What does if \_\_name\_\_ == "\_\_main\_\_" do?</b></summary>

It runs the block only when the file is executed directly, not when it is imported. That lets one file be both a reusable module and a script.

</details>

<details>
<summary><b>2. Why use a virtual environment?</b></summary>

Each project gets its own installed packages and versions, so projects can't break each other, and `requirements.txt` can recreate the exact set on another machine or server.

</details>

<details>
<summary><b>3. json.load vs json.loads?</b></summary>

`loads` parses a string; `load` reads from a file object. Likewise `dumps` returns a string and `dump` writes to a file.

</details>

<details>
<summary><b>4. Which errors should a client retry?</b></summary>

Timeouts, connection errors, 429 and 5xx — with exponential backoff, jitter and a maximum number of attempts. Not other 4xx errors, which mean the request itself is wrong.

</details>

<details>
<summary><b>5. What is pagination and why do APIs use it?</b></summary>

Returning results a page at a time keeps responses small and fast. Offset pagination (page=3) is simple but can skip or repeat rows when data changes; cursor pagination (next=abc) is stable.

</details>

<details>
<summary><b>6. Where should API keys live?</b></summary>

In environment variables or a secrets manager, read with `os.environ`. Never in code, notebooks or git history.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Modules: one file, one job
- [ ] I can explain: Packages and the standard library
- [ ] I can explain: pip and virtual environments
- [ ] I can explain: JSON: the language APIs speak
- [ ] I can explain: Calling a web API
- [ ] I can explain: Robust API clients: pages, retries, limits, caching

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Collect Every Page
- [ ] [Q17](#q17) Validate an API Response
- [ ] [Q18](#q18) Sliding-Window Rate Limiter
- [ ] [Q19](#q19) Cache Repeated API Calls
- [ ] [Q20](#q20) Resolve a Relative Import

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.